# Student Performance Classification

**Tabular Classification Project**

## 2 · Project Overview

Classify **student academic performance** into three categories — Low, Medium, High — based on study habits, demographics, and family background. The dataset has ~2,000 students with balanced target classes.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given student demographics (age, gender, parental education), study habits (study hours, absences, tutoring), and school factors (class size, extracurriculars), predict performance level (Low / Medium / High).

## 5 · Why This Project Matters

- Early identification of struggling students enables **targeted interventions**.
- Education data mining is a growing field with real policy impact.
- Multiclass classification teaches different evaluation from binary problems.
- Understanding which factors drive performance helps allocate resources.
- Ethical considerations: model bias can reinforce educational inequality.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~2,000 |
| Features | 12 (age, gender, parental_education, study_hours_per_week, absences, tutoring, extracurriculars, class_size, internet_access, family_size, travel_time, free_time) |
| Target | `performance` (3-class: 0=Low, 1=Medium, 2=High) |
| Class balance | ~33% each class |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Synthetic student academic dataset for ML practice.
**License**: Educational / public.
**Note**: Inspired by UCI Student Performance and Kaggle student datasets.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "performance"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Student Performance Classification


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 2000

age = np.random.randint(15, 22, n)
gender = np.random.choice(['M', 'F'], n)
parent_edu = np.random.choice(['none', 'primary', 'secondary', 'higher'], n, p=[0.05, 0.2, 0.45, 0.3])
study_hours = np.random.exponential(8, n).clip(0, 40).round(1)
absences = np.random.poisson(5, n).clip(0, 30)
tutoring = np.random.binomial(1, 0.3, n)
extracurriculars = np.random.binomial(1, 0.5, n)
class_size = np.random.choice([20, 25, 30, 35, 40], n)
internet = np.random.binomial(1, 0.75, n)
family_size = np.random.choice(['small', 'large'], n, p=[0.4, 0.6])
travel_time = np.random.choice([1, 2, 3, 4], n, p=[0.4, 0.35, 0.15, 0.1])
free_time = np.random.choice([1, 2, 3, 4, 5], n)

edu_score = {'none': 0, 'primary': 0.5, 'secondary': 1.0, 'higher': 1.5}
score = (
    0.15 * study_hours
    - 0.1 * absences
    + 0.5 * tutoring
    + 0.3 * extracurriculars
    + np.array([edu_score[e] for e in parent_edu])
    + 0.3 * internet
    - 0.05 * travel_time
    + np.random.normal(0, 1.5, n)
)
perf = np.where(score < np.percentile(score, 33), 0,
         np.where(score < np.percentile(score, 67), 1, 2))

df = pd.DataFrame({
    'age': age, 'gender': gender, 'parental_education': parent_edu,
    'study_hours_per_week': study_hours, 'absences': absences, 'tutoring': tutoring,
    'extracurriculars': extracurriculars, 'class_size': class_size,
    'internet_access': internet, 'family_size': family_size,
    'travel_time': travel_time, 'free_time': free_time, 'performance': perf,
})
print(f"Dataset shape: {df.shape}")
print(f"Performance distribution:\n{pd.Series(perf).value_counts().sort_index()}")
df.head()

Dataset shape: (2000, 13)
Performance distribution:
0    660
1    680
2    660
Name: count, dtype: int64


,age,gender,parental_education,study_hours_per_week,absences,tutoring,extracurriculars,class_size,internet_access,family_size,travel_time,free_time,performance
0,21,F,higher,3.1,7,1,1,40,0,large,2,2,1
1,18,M,primary,8.8,7,0,1,30,1,large,2,1,2
2,19,F,higher,6.5,2,0,0,40,1,large,3,1,0
3,21,F,higher,3.4,5,0,1,25,1,large,1,2,0
4,17,M,secondary,1.2,4,0,0,35,0,small,1,3,2


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (2000, 13)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
performance
1    680
2    660
0    660
Name: count, dtype: int64

Dtypes:
age                       int32
gender                   object
parental_education       object
study_hours_per_week    float64
absences                  int32
tutoring                  int32
extracurriculars          int32
class_size                int64
internet_access           int32
family_size              object
travel_time               int64
free_time                 int64
performance               int64
dtype: object

Target 'performance' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(['study_hours_per_week', 'absences', 'age', 'class_size', 'travel_time', 'free_time']):
    ax = axes[i // 3, i % 3]
    for cls in [0, 1, 2]:
        df[df[TARGET]==cls][col].hist(bins=20, ax=ax, alpha=0.5, label=f'Class {cls}')
    ax.set_title(col); ax.legend()
plt.suptitle("Feature Distributions by Performance Class", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(['parental_education', 'family_size']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i])
    axes[i].set_title(f"Performance by {col}")
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 9, Categorical: 3


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['coral', 'gold', 'steelblue'], edgecolor='black')
axes[0].set_title("Student Performance Distribution")
axes[0].set_xticklabels(['Low (0)', 'Medium (1)', 'High (2)'], rotation=0)
df[TARGET].value_counts().sort_index().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['coral', 'gold', 'steelblue'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts().sort_index()}")

Class distribution:
performance
0    660
1    680
2    660
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().sort_index().to_dict()}")

Train: (1600, 12), Test: (400, 12)
Train target dist: {0: 528, 1: 544, 2: 528}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (12): ['age', 'gender', 'parental_education', 'study_hours_per_week', 'absences', 'tutoring', 'extracurriculars', 'class_size', 'internet_access', 'family_size', 'travel_time', 'free_time']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.5125
Precision: 0.5020
Recall   : 0.5125
F1       : 0.5003


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                        
CalibratedClassifierCV           0.5300           0.532828  0.689206  0.509463   0.516654  0.5300    0.103106
LinearSVC                        0.5250           0.527926       NaN  0.502439   0.511109  0.5250    0.028206
RidgeClassifier                  0.5225           0.525327       NaN  0.501804   0.509633  0.5225    0.024387
RidgeClassifierCV                0.5225           0.525327       NaN  0.501804   0.509633  0.5225    0.031216
NearestCentroid                  0.5175           0.519162  0.692286  0.512970   0.516948  0.5175    0.029722
LinearDiscriminantAnalysis       0.5175           0.519088  0.692182  0.513206   0.516572  0.5175    0.030977
LogisticRegression               0.5150           0.517305  0.692615  0.502601   0.504

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
_flaml_metric = "macro_f1" if y_train.nunique() > 2 else "f1"
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric=_flaml_metric,
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: catboost
Accuracy : 0.4975
F1       : 0.4902


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.4700  F1: 0.4647


LightGBM  Acc: 0.4525  F1: 0.4516


XGBoost   Acc: 0.4800  F1: 0.4775


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.5125  0.5003     0.5020  0.5125
FLAML                  0.4975  0.4902     0.4914  0.4975
XGBoost                0.4800  0.4775     0.4779  0.4800
CatBoost               0.4700  0.4647     0.4651  0.4700
LightGBM               0.4525  0.4516     0.4537  0.4525

Top 3: ['Logistic Regression', 'FLAML', 'XGBoost']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  Logistic Regression
              precision    recall  f1-score   support

           0       0.52      0.70      0.60       132
           1       0.39      0.29      0.33       136
           2       0.60      0.56      0.58       132

    accuracy                           0.51       400
   macro avg       0.50      0.51      0.50       400
weighted avg       0.50      0.51      0.50       400


  FLAML
              precision    recall  f1-score   support

           0       0.49      0.62      0.55       132
           1       0.36      0.29      0.32       136
           2       0.63      0.59      0.61       132

    accuracy                           0.50       400
   macro avg       0.49      0.50      0.49       400
weighted avg       0.49      0.50      0.49       400


  XGBoost
              precision    recall  f1-score   support

           0       0.50      0.58      0.53       132
           1       0.37      0.34      0.35 

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: Logistic Regression
Error rate: 0.4875 (195 / 400)

Errors by true class:
      errors  total  error_rate
true                           
0         40    132    0.303030
1         97    136    0.713235
2         58    132    0.439394


## 25 · Interpretation and Business Insight

- **Study hours** is the strongest positive predictor of performance.
- **Absences** negatively impact performance significantly.
- **Parental education** (higher) correlates with better outcomes.
- **Tutoring** and **extracurriculars** contribute positively.
- **Internet access** has a moderate positive effect.
- Demographics (age, gender) are weak predictors.

## 26 · Limitations

1. Synthetic data — real academic performance has many more factors.
2. Only 12 features — missing teacher quality, curriculum, motivation.
3. 3-class target simplifies continuous grade distributions.
4. No longitudinal data (performance over time).
5. Potential proxy discrimination through parental education.

## 27 · How to Improve This Project

1. Collect real student data with proper IRB approval.
2. Add engagement metrics (participation, homework completion).
3. Use ordinal classification (Low < Medium < High).
4. Engineer interaction features (study_hours × tutoring).
5. Apply fairness constraints across demographic groups.

## 28 · Production Considerations

- Early warning system for at-risk students.
- Tool for academic advisors, not automated decisions.
- Must audit for bias across gender, socioeconomic groups.
- Privacy considerations (student data is sensitive).
- Regular revalidation with updated cohorts.

## 29 · Common Mistakes

1. Using model predictions for high-stakes decisions without human review.
2. Not considering fairness across demographic groups.
3. Confusing correlation with causation (parental education ≠ student ability).
4. Ignoring ordinal nature of target (Low < Medium < High).
5. Reporting only accuracy without per-class F1.

## 30 · Mini Challenge / Exercises

1. Compare weighted F1 vs macro F1 — which is more appropriate here?
2. Build a model without demographic features — does fairness improve?
3. Convert to binary (Low vs Medium+High) and compare.
4. Which single feature gives the best accuracy alone?

## 31 · Final Summary / Key Takeaways

1. Student performance depends most on study habits and support systems.
2. Multiclass classification requires per-class evaluation.
3. Parental education is predictive but raises fairness concerns.
4. Early warning systems must balance accuracy with ethical use.
5. Human oversight is essential for educational ML applications.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.47,
    "f1": 0.4647,
    "precision": 0.4651,
    "recall": 0.47
  },
  "LightGBM": {
    "accuracy": 0.4525,
    "f1": 0.4516,
    "precision": 0.4537,
    "recall": 0.4525
  },
  "XGBoost": {
    "accuracy": 0.48,
    "f1": 0.4775,
    "precision": 0.4779,
    "recall": 0.48
  },
  "Logistic Regression": {
    "accuracy": 0.5125,
    "f1": 0.5003,
    "precision": 0.502,
    "recall": 0.5125
  },
  "FLAML": {
    "accuracy": 0.4975,
    "f1": 0.4902,
    "precision": 0.4914,
    "recall": 0.4975
  }
}
